# CSS-Module -> Fluent 2 Migration Notebook

## 1. Konfiguration
Basis-Pfad, Reihenfolge und Dateilisten definieren.

## 2. Python-Helfer
Read/Write/Delete auf OneDrive-Pfaden mit UTF-8.

## 3. CSS-Parser und Mapping-Regeln
Griffel-kompatible Umwandlung inkl. Tokens.

## 4. makeStyles-Block generieren
Klassennamen stabil halten.

## 5. TSX-Transformation
CSS-Import entfernen, Fluent-Importe ergänzen, `const styles = useStyles()` als erste Zeile.

## 6. JSX-Klassenreferenzen
`styles.xxx` beibehalten und an generierte Klassen binden.

## 7. Migration-Lauf pages/

## 8. Migration-Lauf components/editor/

## 9. Migration-Lauf übrige components/

## 10. Migration-Lauf editor/PolygonEditor

## 11. Aufräumen
Restliche `.module.css` und `css-modules.d.ts` prüfen/löschen.

## 12. Validierung
`npx tsc --noEmit` und `npx vitest run` ausführen.

In [1]:
from __future__ import annotations

import re
import subprocess
from collections import OrderedDict
from pathlib import Path

BASE = Path(r"C:/Users/DanielKlas/OneDrive - Tischlermeister Daniel Klas/2_AREAS/AREA_IT_Infrastruktur/Entwicklung/YAKDS/planner-frontend/src")
FRONTEND_ROOT = BASE.parent

PAGES = [
    "pages/S109ShellHarnessPage",
    "pages/CaptureDialogHarnessPage",
    "pages/ConstraintsPanel",
    "pages/KitchenAssistantPanel",
]

EDITOR_COMPONENTS = [
    "components/editor/StatusBar",
    "components/editor/CompassOverlay",
    "components/editor/LockPanel",
    "components/editor/ProtectPanel",
    "components/editor/GroupsPanel",
    "components/editor/MacrosPanel",
    "components/editor/LayoutSheetTabs",
    "components/editor/LevelsPanel",
    "components/editor/DaylightPanel",
    "components/editor/AreasPanel",
    "components/editor/CameraPresetPanel",
    "components/editor/NavigationSettingsPanel",
    "components/editor/RenderEnvironmentPanel",
    "components/editor/SectionPanel",
    "components/editor/StairsPanel",
    "components/editor/VisibilityPanel",
    "components/editor/WallFeaturesPanel",
    "components/editor/RoomFeaturesPanel",
    "components/editor/MaterialPanel",
    "components/editor/Preview3D",
    "components/editor/CanvasArea",
]

OTHER_COMPONENTS = [
    "components/LanguageSwitcher",
    "components/OnboardingWizard",
    "components/business/BusinessPanel",
    "components/catalog/AssetBrowser",
    "components/catalog/AssetImportDialog",
    "components/catalog/CatalogBrowser",
    "components/catalog/MaterialBrowser",
    "components/imports/ImportJobPanel",
    "components/imports/ImportReviewPanel",
    "components/quotes/QuoteExportPanel",
    "components/validation/ValidationPanel",
]

EDITOR = [
    "editor/PolygonEditor",
]

TARGETS = PAGES + EDITOR_COMPONENTS + OTHER_COMPONENTS + EDITOR

VAR_MAP = {
    "--border-subtle": "tokens.colorNeutralStroke2",
    "--border-soft": "tokens.colorNeutralStroke1",
    "--border": "tokens.colorNeutralStroke2",
    "--surface-default": "tokens.colorNeutralBackground1",
    "--surface-muted": "tokens.colorNeutralBackground2",
    "--text-primary": "tokens.colorNeutralForeground1",
    "--text-muted": "tokens.colorNeutralForeground3",
    "--text-inverse": "tokens.colorNeutralForegroundInverted",
    "--text-soft-inverse": "tokens.colorNeutralForegroundInverted",
    "--radius-sm": "tokens.borderRadiusSmall",
    "--radius-md": "tokens.borderRadiusMedium",
    "--radius-lg": "tokens.borderRadiusLarge",
    "--radius-pill": "tokens.borderRadiusCircular",
    "--shadow-sm": "tokens.shadow4",
    "--shadow-card": "tokens.shadow8",
    "--shadow-card-hover": "tokens.shadow16",
    "--primary-color": "tokens.colorBrandForeground1",
    "--primary-light": "tokens.colorBrandBackground2",
    "--primary-hover": "tokens.colorBrandBackground2Hover",
    "--focus-ring": "tokens.colorStrokeFocus2",
    "--status-danger": "tokens.colorPaletteRedForeground1",
    "--status-danger-bg": "tokens.colorPaletteRedBackground1",
    "--status-success": "tokens.colorPaletteGreenBackground3",
    "--status-success-hover": "tokens.colorPaletteGreenBackground3Hover",
    "--status-info": "tokens.colorPaletteBlueForeground1",
    "--status-info-soft": "tokens.colorPaletteBlueBackground1",
    "--overlay-soft": "rgba(0, 0, 0, 0.45)",
}

BORDER_STYLES = {
    "none",
    "hidden",
    "dotted",
    "dashed",
    "solid",
    "double",
    "groove",
    "ridge",
    "inset",
    "outset",
}


def read_utf8(path: Path) -> str:
    return path.read_text(encoding="utf-8")


def write_utf8(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")


def delete_file(path: Path) -> None:
    if path.exists():
        path.unlink()


def js_string(value: str) -> str:
    value = value.replace("\\", "\\\\").replace("'", "\\'")
    return f"'{value}'"


def kebab_to_camel(name: str) -> str:
    parts = name.split("-")
    return parts[0] + "".join(p[:1].upper() + p[1:] for p in parts[1:])


def split_top_level(text: str, sep: str) -> list[str]:
    result: list[str] = []
    current: list[str] = []
    depth_paren = 0
    depth_brack = 0
    in_single = False
    in_double = False
    in_back = False
    i = 0

    while i < len(text):
        ch = text[i]

        if in_single:
            current.append(ch)
            if ch == "'" and text[i - 1] != "\\":
                in_single = False
            i += 1
            continue
        if in_double:
            current.append(ch)
            if ch == '"' and text[i - 1] != "\\":
                in_double = False
            i += 1
            continue
        if in_back:
            current.append(ch)
            if ch == "`" and text[i - 1] != "\\":
                in_back = False
            i += 1
            continue

        if ch == "'":
            in_single = True
            current.append(ch)
        elif ch == '"':
            in_double = True
            current.append(ch)
        elif ch == "`":
            in_back = True
            current.append(ch)
        elif ch == "(":
            depth_paren += 1
            current.append(ch)
        elif ch == ")":
            depth_paren = max(0, depth_paren - 1)
            current.append(ch)
        elif ch == "[":
            depth_brack += 1
            current.append(ch)
        elif ch == "]":
            depth_brack = max(0, depth_brack - 1)
            current.append(ch)
        elif ch == sep and depth_paren == 0 and depth_brack == 0:
            result.append("".join(current))
            current = []
        else:
            current.append(ch)
        i += 1

    result.append("".join(current))
    return result


def map_css_var(var_name: str) -> str:
    mapped = VAR_MAP.get(var_name)
    if mapped is None:
        return js_string(f"var({var_name})")
    if mapped.startswith("tokens."):
        return mapped
    return js_string(mapped)


def value_to_js(value: str) -> str:
    value = " ".join(value.strip().split())
    if not value:
        return js_string("")

    vars_found = list(re.finditer(r"var\((--[a-zA-Z0-9\-_]+)\)", value))
    if not vars_found:
        return js_string(value)

    if len(vars_found) == 1 and vars_found[0].span() == (0, len(value)):
        return map_css_var(vars_found[0].group(1))

    out: list[str] = []
    has_expr = False
    last = 0
    for m in vars_found:
        if m.start() > last:
            out.append(value[last:m.start()])
        mapped = map_css_var(m.group(1))
        if mapped.startswith("tokens."):
            has_expr = True
            out.append("${" + mapped + "}")
        else:
            out.append(mapped[1:-1])
        last = m.end()
    if last < len(value):
        out.append(value[last:])

    template = "".join(out).replace("`", "\\`")
    if has_expr:
        return f"`{template}`"
    return js_string(template)


def extract_block(text: str, start_brace_idx: int) -> tuple[str, int]:
    depth = 0
    i = start_brace_idx
    while i < len(text):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start_brace_idx + 1 : i], i + 1
        i += 1
    raise ValueError("Unterminated block")


def parse_declarations(body: str) -> list[tuple[str, str]]:
    decls: list[tuple[str, str]] = []
    for part in split_top_level(body, ";"):
        if ":" not in part:
            continue
        prop, val = part.split(":", 1)
        prop = prop.strip()
        val = val.strip()
        if prop and val:
            decls.append((prop, val))
    return decls


def parse_css_rules(css: str, media: str | None = None) -> list[tuple[str, list[tuple[str, str]], str | None]]:
    rules: list[tuple[str, list[tuple[str, str]], str | None]] = []
    i = 0
    n = len(css)
    while i < n:
        while i < n and css[i].isspace():
            i += 1
        if i >= n:
            break

        if css.startswith("@media", i):
            brace = css.find("{", i)
            if brace == -1:
                break
            media_query = css[i:brace].strip()
            inner, end_idx = extract_block(css, brace)
            rules.extend(parse_css_rules(inner, media_query))
            i = end_idx
            continue

        if css[i] == "@":
            semi = css.find(";", i)
            brace = css.find("{", i)
            if semi != -1 and (brace == -1 or semi < brace):
                i = semi + 1
                continue
            if brace != -1:
                _, end_idx = extract_block(css, brace)
                i = end_idx
                continue
            break

        brace = css.find("{", i)
        if brace == -1:
            break
        selector = css[i:brace].strip()
        body, end_idx = extract_block(css, brace)
        decls = parse_declarations(body)
        if selector and decls:
            rules.append((selector, decls, media))
        i = end_idx

    return rules


def ensure_nested(target: OrderedDict, key: str) -> OrderedDict:
    existing = target.get(key)
    if isinstance(existing, dict):
        return existing
    nested: OrderedDict = OrderedDict()
    target[key] = nested
    return nested


def normalize_nested_selector(rem: str) -> str:
    rem = rem.rstrip()
    if not rem:
        return ""
    if rem[0] in [":", "[", ".", ">", "+", "~"]:
        return "&" + rem
    if rem[0].isspace():
        return "&" + rem
    return "& " + rem


def convert_decl_block(decls: list[tuple[str, str]]) -> OrderedDict:
    out: OrderedDict = OrderedDict()
    border_value = None
    border_width = None
    border_style = None
    border_color_expr = None

    for prop, raw in decls:
        prop_lower = prop.lower()
        if prop_lower == "border":
            border_value = raw
        elif prop_lower == "border-width":
            border_width = raw
        elif prop_lower == "border-style":
            border_style = raw
        elif prop_lower == "border-color":
            border_color_expr = value_to_js(raw)
            continue

        key = kebab_to_camel(prop_lower)
        out[key] = value_to_js(raw)

    if border_color_expr is not None:
        width = "1px"
        style = "solid"
        if border_value:
            for token in border_value.split():
                if token.endswith("px") or token in {"thin", "medium", "thick", "0"}:
                    width = token
                if token in BORDER_STYLES:
                    style = token
        if border_width:
            width = border_width
        if border_style:
            style = border_style

        if border_color_expr.startswith("tokens."):
            out["border"] = f"`{width} {style} ${{{border_color_expr}}}`"
        elif border_color_expr.startswith("`") and border_color_expr.endswith("`"):
            out["border"] = f"`{width} {style} {border_color_expr[1:-1]}`"
        elif border_color_expr.startswith("'") and border_color_expr.endswith("'"):
            out["border"] = js_string(f"{width} {style} {border_color_expr[1:-1]}")
        else:
            out["border"] = js_string(f"{width} {style} {border_color_expr}")

        if "borderColor" in out:
            del out["borderColor"]

    return out


def build_make_styles(css_text: str) -> str:
    css_text = re.sub(r"/\*.*?\*/", "", css_text, flags=re.S)
    rules = parse_css_rules(css_text)
    classes: OrderedDict[str, OrderedDict] = OrderedDict()

    for selector_group, decls, media in rules:
        props = convert_decl_block(decls)
        selectors = [s.strip() for s in split_top_level(selector_group, ",") if s.strip()]
        for selector in selectors:
            if not selector.startswith("."):
                continue
            m = re.match(r"^\.([A-Za-z0-9_-]+)(.*)$", selector)
            if not m:
                continue
            class_name = m.group(1)
            rem = m.group(2)

            if class_name not in classes:
                classes[class_name] = OrderedDict()
            target = classes[class_name]

            if media:
                target = ensure_nested(target, media)

            if rem.rstrip():
                nested = normalize_nested_selector(rem)
                target = ensure_nested(target, nested)

            for k, v in props.items():
                target[k] = v

    def render_obj(obj: OrderedDict, indent: int) -> str:
        lines = []
        pad = " " * indent
        for key, value in obj.items():
            key_render = key if re.match(r"^[A-Za-z_$][A-Za-z0-9_$]*$", key) else js_string(key)
            if isinstance(value, dict):
                lines.append(f"{pad}{key_render}: {{")
                lines.append(render_obj(value, indent + 2))
                lines.append(f"{pad}}},")
            else:
                lines.append(f"{pad}{key_render}: {value},")
        return "\n".join(lines)

    return "const useStyles = makeStyles({\n" + render_obj(classes, 2) + "\n})\n"


def ensure_fluent_import(src: str) -> str:
    fluent_re = re.compile(r"^import[\s\S]*?from\s+['\"]@fluentui/react-components['\"]\n?", re.M)
    m = fluent_re.search(src)
    if m:
        stmt = m.group(0)
        brace_m = re.search(r"\{([\s\S]*?)\}", stmt)
        names = []
        if brace_m:
            names = [n.strip() for n in brace_m.group(1).split(",") if n.strip()]
        name_set = set(names)
        name_set.add("makeStyles")
        name_set.add("tokens")
        rebuilt = "import { " + ", ".join(sorted(name_set)) + " } from '@fluentui/react-components'\n"
        return src[: m.start()] + rebuilt + src[m.end() :]

    import_re = re.compile(r"^import[\s\S]*?from\s+['\"][^'\"]+['\"]\n?", re.M)
    imports = list(import_re.finditer(src))
    insert_at = imports[-1].end() if imports else 0
    return src[:insert_at] + "import { makeStyles, tokens } from '@fluentui/react-components'\n" + src[insert_at:]


def find_matching_brace(text: str, open_idx: int) -> int:
    depth = 0
    i = open_idx
    in_single = False
    in_double = False
    in_back = False
    in_line_comment = False
    in_block_comment = False

    while i < len(text):
        ch = text[i]
        nxt = text[i + 1] if i + 1 < len(text) else ""

        if in_line_comment:
            if ch == "\n":
                in_line_comment = False
            i += 1
            continue
        if in_block_comment:
            if ch == "*" and nxt == "/":
                in_block_comment = False
                i += 2
                continue
            i += 1
            continue
        if in_single:
            if ch == "'" and text[i - 1] != "\\":
                in_single = False
            i += 1
            continue
        if in_double:
            if ch == '"' and text[i - 1] != "\\":
                in_double = False
            i += 1
            continue
        if in_back:
            if ch == "`" and text[i - 1] != "\\":
                in_back = False
            i += 1
            continue

        if ch == "/" and nxt == "/":
            in_line_comment = True
            i += 2
            continue
        if ch == "/" and nxt == "*":
            in_block_comment = True
            i += 2
            continue
        if ch == "'":
            in_single = True
            i += 1
            continue
        if ch == '"':
            in_double = True
            i += 1
            continue
        if ch == "`":
            in_back = True
            i += 1
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return i
        i += 1

    return -1


def insert_use_styles_calls(src: str) -> tuple[str, int]:
    patterns = [
        re.compile(r"(?:export\s+)?function\s+[A-Za-z0-9_]+\s*\([^\)]*\)\s*\{"),
        re.compile(r"(?:export\s+)?const\s+[A-Za-z0-9_]+[^=\n]*=\s*(?:async\s*)?\([^\)]*\)\s*=>\s*\{"),
    ]

    matches = []
    for pattern in patterns:
        matches.extend(list(pattern.finditer(src)))
    matches.sort(key=lambda m: m.start())

    inserts = []
    for m in matches:
        open_idx = src.find("{", m.start(), m.end())
        if open_idx == -1:
            continue
        close_idx = find_matching_brace(src, open_idx)
        if close_idx == -1:
            continue

        body = src[open_idx + 1 : close_idx]
        if "styles." not in body:
            continue
        if re.search(r"\bconst\s+styles\s*=\s*useStyles\s*\(\s*\)", body):
            continue

        line_start = src.rfind("\n", 0, open_idx) + 1
        fn_indent = re.match(r"[ \t]*", src[line_start:open_idx]).group(0)
        inner_indent = fn_indent + "  "
        insertion = "\n" + inner_indent + "const styles = useStyles();\n"
        inserts.append((open_idx + 1, insertion))

    inserts.sort(key=lambda item: item[0], reverse=True)
    for pos, text in inserts:
        src = src[:pos] + text + src[pos:]

    return src, len(inserts)


def migrate_pair(base_rel: str) -> tuple[bool, str]:
    tsx_path = BASE / f"{base_rel}.tsx"
    css_path = BASE / f"{base_rel}.module.css"

    if not tsx_path.exists() or not css_path.exists():
        return False, f"SKIP missing: {base_rel}"

    tsx_src = read_utf8(tsx_path)
    css_src = read_utf8(css_path)

    tsx_src = re.sub(
        r"^import\s+styles\s+from\s+['\"][^'\"]+\.module\.css['\"]\s*\n?",
        "",
        tsx_src,
        flags=re.M,
    )

    tsx_src = ensure_fluent_import(tsx_src)
    styles_block = build_make_styles(css_src)

    import_re = re.compile(r"^import[\s\S]*?from\s+['\"][^'\"]+['\"]\n?", re.M)
    imports = list(import_re.finditer(tsx_src))
    insert_at = imports[-1].end() if imports else 0

    prefix = tsx_src[:insert_at]
    suffix = tsx_src[insert_at:]
    if not prefix.endswith("\n"):
        prefix += "\n"
    tsx_src = prefix + "\n" + styles_block + "\n" + suffix.lstrip("\n")

    tsx_src, inserted = insert_use_styles_calls(tsx_src)

    write_utf8(tsx_path, tsx_src)
    delete_file(css_path)
    return True, f"MIGRATED {base_rel} (useStyles insertions: {inserted})"


report_lines = []
report_lines.append(f"Target count: {len(TARGETS)}")
ok_count = 0
for rel in TARGETS:
    ok, msg = migrate_pair(rel)
    report_lines.append(msg)
    if ok:
        ok_count += 1

remaining_css = sorted(p.relative_to(BASE).as_posix() for p in BASE.rglob("*.module.css"))
report_lines.append(f"Migrated count: {ok_count}")
report_lines.append(f"Remaining .module.css: {len(remaining_css)}")
for p in remaining_css:
    report_lines.append(f"REMAIN {p}")

if not remaining_css:
    css_dts = BASE / "css-modules.d.ts"
    if css_dts.exists():
        delete_file(css_dts)
        report_lines.append("Deleted css-modules.d.ts")

# Validation commands

tsc_log = FRONTEND_ROOT / "migration_tsc.log"
vitest_log = FRONTEND_ROOT / "migration_vitest.log"

tsc = subprocess.run(
    ["npx", "tsc", "--noEmit"],
    cwd=str(FRONTEND_ROOT),
    capture_output=True,
    text=True,
)
write_utf8(tsc_log, (tsc.stdout or "") + "\n" + (tsc.stderr or ""))
report_lines.append(f"tsc exit: {tsc.returncode}")

vitest = subprocess.run(
    ["npx", "vitest", "run"],
    cwd=str(FRONTEND_ROOT),
    capture_output=True,
    text=True,
)
write_utf8(vitest_log, (vitest.stdout or "") + "\n" + (vitest.stderr or ""))
report_lines.append(f"vitest exit: {vitest.returncode}")

report_path = FRONTEND_ROOT / "migration_notebook_report.txt"
write_utf8(report_path, "\n".join(report_lines) + "\n")

print("Done. Report:", report_path)
print("tsc log:", tsc_log)
print("vitest log:", vitest_log)

FileNotFoundError: [WinError 2] Das System kann die angegebene Datei nicht finden

In [2]:
from __future__ import annotations

import re
import subprocess
from collections import OrderedDict
from pathlib import Path

BASE = Path(r"C:/Users/DanielKlas/OneDrive - Tischlermeister Daniel Klas/2_AREAS/AREA_IT_Infrastruktur/Entwicklung/YAKDS/planner-frontend/src")
REPO_ROOT = BASE.parent.parent

TARGETS = [
    "pages/S109ShellHarnessPage",
    "pages/CaptureDialogHarnessPage",
    "pages/ConstraintsPanel",
    "pages/KitchenAssistantPanel",
    "components/editor/StatusBar",
    "components/editor/CompassOverlay",
    "components/editor/LockPanel",
    "components/editor/ProtectPanel",
    "components/editor/GroupsPanel",
    "components/editor/MacrosPanel",
    "components/editor/LayoutSheetTabs",
    "components/editor/LevelsPanel",
    "components/editor/DaylightPanel",
    "components/editor/AreasPanel",
    "components/editor/CameraPresetPanel",
    "components/editor/NavigationSettingsPanel",
    "components/editor/RenderEnvironmentPanel",
    "components/editor/SectionPanel",
    "components/editor/StairsPanel",
    "components/editor/VisibilityPanel",
    "components/editor/WallFeaturesPanel",
    "components/editor/RoomFeaturesPanel",
    "components/editor/MaterialPanel",
    "components/editor/Preview3D",
    "components/editor/CanvasArea",
    "components/LanguageSwitcher",
    "components/OnboardingWizard",
    "components/business/BusinessPanel",
    "components/catalog/AssetBrowser",
    "components/catalog/AssetImportDialog",
    "components/catalog/CatalogBrowser",
    "components/catalog/MaterialBrowser",
    "components/imports/ImportJobPanel",
    "components/imports/ImportReviewPanel",
    "components/quotes/QuoteExportPanel",
    "components/validation/ValidationPanel",
    "editor/PolygonEditor",
]

restore_files = []
for rel in TARGETS:
    restore_files.append(str((BASE / f"{rel}.tsx").relative_to(REPO_ROOT)).replace('\\\\', '/'))
    restore_files.append(str((BASE / f"{rel}.module.css").relative_to(REPO_ROOT)).replace('\\\\', '/'))
restore_files.append(str((BASE / "css-modules.d.ts").relative_to(REPO_ROOT)).replace('\\\\', '/'))

subprocess.run(["git", "restore", "--"] + restore_files, cwd=str(REPO_ROOT), check=True)

VAR_MAP = {
    "--border-subtle": "tokens.colorNeutralStroke2",
    "--border-soft": "tokens.colorNeutralStroke1",
    "--border": "tokens.colorNeutralStroke2",
    "--surface-default": "tokens.colorNeutralBackground1",
    "--surface-muted": "tokens.colorNeutralBackground2",
    "--text-primary": "tokens.colorNeutralForeground1",
    "--text-muted": "tokens.colorNeutralForeground3",
    "--text-inverse": "tokens.colorNeutralForegroundInverted",
    "--text-soft-inverse": "tokens.colorNeutralForegroundInverted",
    "--radius-sm": "tokens.borderRadiusSmall",
    "--radius-md": "tokens.borderRadiusMedium",
    "--radius-lg": "tokens.borderRadiusLarge",
    "--radius-pill": "tokens.borderRadiusCircular",
    "--shadow-sm": "tokens.shadow4",
    "--shadow-card": "tokens.shadow8",
    "--shadow-card-hover": "tokens.shadow16",
    "--primary-color": "tokens.colorBrandForeground1",
    "--primary-light": "tokens.colorBrandBackground2",
    "--primary-hover": "tokens.colorBrandBackground2Hover",
    "--focus-ring": "tokens.colorStrokeFocus2",
    "--status-danger": "tokens.colorPaletteRedForeground1",
    "--status-danger-bg": "tokens.colorPaletteRedBackground1",
    "--status-success": "tokens.colorPaletteGreenForeground1",
    "--status-success-hover": "tokens.colorPaletteGreenBackground3Hover",
    "--status-info": "tokens.colorPaletteBlueForeground1",
    "--status-info-soft": "tokens.colorPaletteBlueBackground1",
    "--overlay-soft": "rgba(0, 0, 0, 0.45)",
}


def js_str(v: str) -> str:
    return "'" + v.replace('\\', '\\\\').replace("'", "\\'") + "'"


def val_js(v: str) -> str:
    v = ' '.join(v.strip().split())
    parts = re.split(r"(var\(--[a-zA-Z0-9\-_]+\))", v)
    if len(parts) == 1:
        return js_str(v)
    out = []
    has_expr = False
    for p in parts:
        if p.startswith('var(--') and p.endswith(')'):
            name = p[4:-1]
            mapped = VAR_MAP.get(name)
            if mapped and mapped.startswith('tokens.'):
                out.append('${' + mapped + '}')
                has_expr = True
            elif mapped:
                out.append(mapped)
            else:
                out.append(p)
        else:
            out.append(p)
    merged = ''.join(out).replace('`', '\\`')
    if has_expr:
        if re.fullmatch(r"\$\{tokens\.[^}]+\}", merged):
            return merged[2:-1]
        return '`' + merged + '`'
    return js_str(merged)


def to_camel(k: str) -> str:
    p = k.split('-')
    return p[0] + ''.join(s[:1].upper() + s[1:] for s in p[1:])


def split_top(s: str, sep: str):
    out, cur, d = [], [], 0
    for ch in s:
        if ch == '(':
            d += 1
        elif ch == ')':
            d = max(0, d - 1)
        if ch == sep and d == 0:
            out.append(''.join(cur))
            cur = []
        else:
            cur.append(ch)
    out.append(''.join(cur))
    return out


def extract_block(s: str, i: int):
    d = 0
    j = i
    while j < len(s):
        if s[j] == '{':
            d += 1
        elif s[j] == '}':
            d -= 1
            if d == 0:
                return s[i + 1:j], j + 1
        j += 1
    raise ValueError('block')


def parse_css(css: str):
    css = re.sub(r"/\*.*?\*/", '', css, flags=re.S)
    rules = []
    i = 0
    while i < len(css):
        while i < len(css) and css[i].isspace():
            i += 1
        if i >= len(css):
            break
        if css.startswith('@media', i):
            b = css.find('{', i)
            m = css[i:b].strip()
            inner, ni = extract_block(css, b)
            for sel, decls, _ in parse_css(inner):
                rules.append((sel, decls, m))
            i = ni
            continue
        if css[i] == '@':
            semi = css.find(';', i)
            if semi == -1:
                break
            i = semi + 1
            continue
        b = css.find('{', i)
        if b == -1:
            break
        sel = css[i:b].strip()
        body, ni = extract_block(css, b)
        decls = []
        for d in split_top(body, ';'):
            if ':' in d:
                k, v = d.split(':', 1)
                k, v = k.strip(), v.strip()
                if k and v:
                    decls.append((k, v))
        if sel and decls:
            rules.append((sel, decls, None))
        i = ni
    return rules


def make_styles(css: str) -> str:
    obj: OrderedDict[str, OrderedDict] = OrderedDict()
    for sels, decls, media in parse_css(css):
        for sel in [x.strip() for x in sels.split(',') if x.strip()]:
            m = re.match(r"^\.([A-Za-z0-9_-]+)(.*)$", sel)
            if not m:
                continue
            cls, rest = m.group(1), m.group(2)
            o = obj.setdefault(cls, OrderedDict())
            tgt = o
            if media:
                tgt = tgt.setdefault(media, OrderedDict())
            rest = rest.rstrip()
            if rest:
                key = '&' + rest if rest.startswith((':', '[', '.', ' ', '>', '+', '~')) else '& ' + rest
                tgt = tgt.setdefault(key, OrderedDict())
            border_color = None
            for k, v in decls:
                if k.lower() == 'border-color':
                    border_color = val_js(v)
                    continue
                tgt[to_camel(k.lower())] = val_js(v)
            if border_color:
                if border_color.startswith('tokens.'):
                    tgt['border'] = '`1px solid ${' + border_color + '}`'
                else:
                    tgt['border'] = js_str('1px solid ' + border_color.strip("'`"))

    def render(d: OrderedDict, ind=2):
        lines = []
        p = ' ' * ind
        for k, v in d.items():
            kk = k if re.match(r"^[A-Za-z_$][A-Za-z0-9_$]*$", k) else js_str(k)
            if isinstance(v, dict):
                lines.append(f"{p}{kk}: {{")
                lines.extend(render(v, ind + 2))
                lines.append(f"{p}}},")
            else:
                lines.append(f"{p}{kk}: {v},")
        return lines

    return 'const useStyles = makeStyles({\n' + '\n'.join(render(obj)) + '\n})\n'


def add_fluent_import(src: str) -> str:
    fluent_line = re.compile(r"^import\s+\{([^}]*)\}\s+from\s+['\"]@fluentui/react-components['\"]\s*$", re.M)
    m = fluent_line.search(src)
    if m:
        names = {x.strip() for x in m.group(1).split(',') if x.strip()}
        names.update({'makeStyles', 'tokens'})
        rep = "import { " + ', '.join(sorted(names)) + " } from '@fluentui/react-components'"
        return src[:m.start()] + rep + src[m.end():]
    imports = list(re.finditer(r"^import .*?(?:\n(?:\s+.*\n)*)?", src, re.M))
    pos = imports[-1].end() if imports else 0
    return src[:pos] + "\nimport { makeStyles, tokens } from '@fluentui/react-components'\n" + src[pos:]


def find_functions(src: str):
    pats = [
        re.compile(r"(?:export\s+)?function\s+[A-Za-z0-9_]+\s*\([^)]*\)\s*\{"),
        re.compile(r"(?:export\s+)?const\s+[A-Za-z0-9_]+[^=\n]*=\s*(?:async\s*)?\([^)]*\)\s*=>\s*\{"),
    ]
    ms = []
    for p in pats:
        ms.extend(list(p.finditer(src)))
    return sorted(ms, key=lambda x: x.start())


def match_brace(src: str, i: int):
    d = 0
    j = i
    while j < len(src):
        if src[j] == '{':
            d += 1
        elif src[j] == '}':
            d -= 1
            if d == 0:
                return j
        j += 1
    return -1


def inject_use_styles(src: str) -> str:
    inserts = []
    for m in find_functions(src):
        o = src.find('{', m.start(), m.end())
        c = match_brace(src, o)
        if o == -1 or c == -1:
            continue
        body = src[o + 1:c]
        if 'styles.' not in body:
            continue
        if re.search(r"\bconst\s+styles\s*=\s*useStyles\(\)", body):
            continue
        ls = src.rfind('\n', 0, o) + 1
        ind = re.match(r"[ \t]*", src[ls:o]).group(0) + '  '
        inserts.append((o + 1, '\n' + ind + 'const styles = useStyles();\n'))
    for p, t in sorted(inserts, key=lambda x: x[0], reverse=True):
        src = src[:p] + t + src[p:]
    return src


for rel in TARGETS:
    tsx = BASE / f"{rel}.tsx"
    css = BASE / f"{rel}.module.css"
    if not tsx.exists() or not css.exists():
        continue
    tsx_src = tsx.read_text(encoding='utf-8')
    css_src = css.read_text(encoding='utf-8')

    tsx_src = re.sub(r"^import\s+styles\s+from\s+['\"].*?\.module\.css['\"]\s*\n?", '', tsx_src, flags=re.M)
    tsx_src = add_fluent_import(tsx_src)
    block = make_styles(css_src)

    imports = list(re.finditer(r"^import .*?(?:\n(?:\s+.*\n)*)?", tsx_src, re.M))
    pos = imports[-1].end() if imports else 0
    tsx_src = tsx_src[:pos] + '\n\n' + block + '\n' + tsx_src[pos:].lstrip('\n')
    tsx_src = inject_use_styles(tsx_src)

    tsx.write_text(tsx_src, encoding='utf-8')
    css.unlink()

remaining = list(BASE.rglob('*.module.css'))
if not remaining:
    dts = BASE / 'css-modules.d.ts'
    if dts.exists():
        dts.unlink()

print('Remaining CSS modules:', len(remaining))

Remaining CSS modules: 0


In [3]:
from pathlib import Path
import re

base = Path(r"C:/Users/DanielKlas/OneDrive - Tischlermeister Daniel Klas/2_AREAS/AREA_IT_Infrastruktur/Entwicklung/YAKDS/planner-frontend/src")

targets = [
    "pages/S109ShellHarnessPage.tsx","pages/CaptureDialogHarnessPage.tsx","pages/ConstraintsPanel.tsx","pages/KitchenAssistantPanel.tsx",
    "components/editor/StatusBar.tsx","components/editor/CompassOverlay.tsx","components/editor/LockPanel.tsx","components/editor/ProtectPanel.tsx",
    "components/editor/GroupsPanel.tsx","components/editor/MacrosPanel.tsx","components/editor/LayoutSheetTabs.tsx","components/editor/LevelsPanel.tsx",
    "components/editor/DaylightPanel.tsx","components/editor/AreasPanel.tsx","components/editor/CameraPresetPanel.tsx","components/editor/NavigationSettingsPanel.tsx",
    "components/editor/RenderEnvironmentPanel.tsx","components/editor/SectionPanel.tsx","components/editor/StairsPanel.tsx","components/editor/VisibilityPanel.tsx",
    "components/editor/WallFeaturesPanel.tsx","components/editor/RoomFeaturesPanel.tsx","components/editor/MaterialPanel.tsx","components/editor/Preview3D.tsx",
    "components/editor/CanvasArea.tsx","components/LanguageSwitcher.tsx","components/OnboardingWizard.tsx","components/business/BusinessPanel.tsx",
    "components/catalog/AssetBrowser.tsx","components/catalog/AssetImportDialog.tsx","components/catalog/CatalogBrowser.tsx","components/catalog/MaterialBrowser.tsx",
    "components/imports/ImportJobPanel.tsx","components/imports/ImportReviewPanel.tsx","components/quotes/QuoteExportPanel.tsx","components/validation/ValidationPanel.tsx",
    "editor/PolygonEditor.tsx"
]

dupes=[]
for rel in targets:
    p = base / rel
    src = p.read_text(encoding='utf-8')
    cnt = len(re.findall(r"const\s+useStyles\s*=\s*makeStyles\(", src))
    if cnt != 1:
        dupes.append((rel,cnt))

print("files with useStyles count != 1:", dupes)


files with useStyles count != 1: []


In [4]:
from __future__ import annotations

import re
import subprocess
from collections import OrderedDict
from pathlib import Path

BASE = Path(r"C:/Users/DanielKlas/OneDrive - Tischlermeister Daniel Klas/2_AREAS/AREA_IT_Infrastruktur/Entwicklung/YAKDS/planner-frontend/src")
REPO = BASE.parent.parent

TARGETS = [
    "pages/S109ShellHarnessPage","pages/CaptureDialogHarnessPage","pages/ConstraintsPanel","pages/KitchenAssistantPanel",
    "components/editor/StatusBar","components/editor/CompassOverlay","components/editor/LockPanel","components/editor/ProtectPanel",
    "components/editor/GroupsPanel","components/editor/MacrosPanel","components/editor/LayoutSheetTabs","components/editor/LevelsPanel",
    "components/editor/DaylightPanel","components/editor/AreasPanel","components/editor/CameraPresetPanel","components/editor/NavigationSettingsPanel",
    "components/editor/RenderEnvironmentPanel","components/editor/SectionPanel","components/editor/StairsPanel","components/editor/VisibilityPanel",
    "components/editor/WallFeaturesPanel","components/editor/RoomFeaturesPanel","components/editor/MaterialPanel","components/editor/Preview3D",
    "components/editor/CanvasArea","components/LanguageSwitcher","components/OnboardingWizard","components/business/BusinessPanel",
    "components/catalog/AssetBrowser","components/catalog/AssetImportDialog","components/catalog/CatalogBrowser","components/catalog/MaterialBrowser",
    "components/imports/ImportJobPanel","components/imports/ImportReviewPanel","components/quotes/QuoteExportPanel","components/validation/ValidationPanel",
    "editor/PolygonEditor",
]

restore = []
for rel in TARGETS:
    restore.append(str((BASE / f"{rel}.tsx").relative_to(REPO)).replace('\\\\', '/'))
    restore.append(str((BASE / f"{rel}.module.css").relative_to(REPO)).replace('\\\\', '/'))
restore.append(str((BASE / "css-modules.d.ts").relative_to(REPO)).replace('\\\\', '/'))
subprocess.run(["git", "restore", "--"] + restore, cwd=str(REPO), check=True)

VAR_MAP = {
    "--border-subtle": "tokens.colorNeutralStroke2",
    "--border-soft": "tokens.colorNeutralStroke1",
    "--border": "tokens.colorNeutralStroke2",
    "--surface-default": "tokens.colorNeutralBackground1",
    "--surface-muted": "tokens.colorNeutralBackground2",
    "--text-primary": "tokens.colorNeutralForeground1",
    "--text-muted": "tokens.colorNeutralForeground3",
    "--text-inverse": "tokens.colorNeutralForegroundInverted",
    "--text-soft-inverse": "tokens.colorNeutralForegroundInverted",
    "--radius-sm": "tokens.borderRadiusSmall",
    "--radius-md": "tokens.borderRadiusMedium",
    "--radius-lg": "tokens.borderRadiusLarge",
    "--radius-pill": "tokens.borderRadiusCircular",
    "--shadow-sm": "tokens.shadow4",
    "--shadow-card": "tokens.shadow8",
    "--shadow-card-hover": "tokens.shadow16",
    "--primary-color": "tokens.colorBrandForeground1",
    "--primary-light": "tokens.colorBrandBackground2",
    "--primary-hover": "tokens.colorBrandBackground2Hover",
    "--focus-ring": "tokens.colorStrokeFocus2",
    "--status-danger": "tokens.colorPaletteRedForeground1",
    "--status-danger-bg": "tokens.colorPaletteRedBackground1",
    "--status-success": "tokens.colorPaletteGreenForeground1",
    "--status-success-hover": "tokens.colorPaletteGreenBackground3Hover",
    "--status-info": "tokens.colorPaletteBlueForeground1",
    "--status-info-soft": "tokens.colorPaletteBlueBackground1",
    "--overlay-soft": "rgba(0, 0, 0, 0.45)",
}


def js(v: str) -> str:
    return "'" + v.replace('\\', '\\\\').replace("'", "\\'") + "'"


def val(v: str) -> str:
    v = ' '.join(v.strip().split())
    bits = re.split(r"(var\(--[a-zA-Z0-9\-_]+\))", v)
    if len(bits) == 1:
        return js(v)
    out = []
    has = False
    for b in bits:
        if b.startswith('var(--') and b.endswith(')'):
            m = VAR_MAP.get(b[4:-1])
            if m and m.startswith('tokens.'):
                out.append('${' + m + '}')
                has = True
            elif m:
                out.append(m)
            else:
                out.append(b)
        else:
            out.append(b)
    t = ''.join(out).replace('`', '\\`')
    if has:
        if re.fullmatch(r"\$\{tokens\.[^}]+\}", t):
            return t[2:-1]
        return '`' + t + '`'
    return js(t)


def camel(k: str) -> str:
    p = k.split('-')
    return p[0] + ''.join(s[:1].upper() + s[1:] for s in p[1:])


def split_top(s: str, sep: str):
    o, c, d = [], [], 0
    for ch in s:
        if ch == '(':
            d += 1
        elif ch == ')':
            d = max(0, d - 1)
        if ch == sep and d == 0:
            o.append(''.join(c)); c = []
        else:
            c.append(ch)
    o.append(''.join(c))
    return o


def block(s: str, i: int):
    d = 0
    j = i
    while j < len(s):
        if s[j] == '{': d += 1
        elif s[j] == '}':
            d -= 1
            if d == 0:
                return s[i+1:j], j+1
        j += 1
    raise RuntimeError('block')


def parse(css: str):
    css = re.sub(r"/\*.*?\*/", '', css, flags=re.S)
    r, i = [], 0
    while i < len(css):
        while i < len(css) and css[i].isspace(): i += 1
        if i >= len(css): break
        if css.startswith('@media', i):
            b = css.find('{', i)
            m = css[i:b].strip()
            inner, ni = block(css, b)
            for s, d, _ in parse(inner):
                r.append((s, d, m))
            i = ni
            continue
        if css[i] == '@':
            semi = css.find(';', i)
            if semi == -1: break
            i = semi + 1
            continue
        b = css.find('{', i)
        if b == -1: break
        sel = css[i:b].strip()
        body, ni = block(css, b)
        ds = []
        for p in split_top(body, ';'):
            if ':' in p:
                k, v = p.split(':', 1)
                k, v = k.strip(), v.strip()
                if k and v: ds.append((k, v))
        if sel and ds: r.append((sel, ds, None))
        i = ni
    return r


def make_block(css: str) -> str:
    data: OrderedDict[str, OrderedDict] = OrderedDict()
    for sels, ds, media in parse(css):
        for sel in [x.strip() for x in sels.split(',') if x.strip()]:
            m = re.match(r"^\.([A-Za-z0-9_-]+)(.*)$", sel)
            if not m: continue
            cls, rest = m.group(1), m.group(2).rstrip()
            root = data.setdefault(cls, OrderedDict())
            tgt = root.setdefault(media, OrderedDict()) if media else root
            if rest:
                key = '&' + rest if rest.startswith((':', '[', '.', ' ', '>', '+', '~')) else '& ' + rest
                tgt = tgt.setdefault(key, OrderedDict())
            bcol = None
            for k, v in ds:
                if k.lower() == 'border-color':
                    bcol = val(v)
                    continue
                tgt[camel(k.lower())] = val(v)
            if bcol:
                tgt['border'] = '`1px solid ${' + bcol + '}`' if bcol.startswith('tokens.') else js('1px solid ' + bcol.strip("'`"))

    def render(d: OrderedDict, ind=2):
        lines = []
        p = ' ' * ind
        for k, v in d.items():
            kk = k if re.match(r"^[A-Za-z_$][A-Za-z0-9_$]*$", k) else js(k)
            if isinstance(v, dict):
                lines.append(f"{p}{kk}: {{")
                lines.extend(render(v, ind + 2))
                lines.append(f"{p}}},")
            else:
                lines.append(f"{p}{kk}: {v},")
        return lines

    return 'const useStyles = makeStyles({\n' + '\n'.join(render(data)) + '\n})\n'


def import_block_end(lines: list[str]) -> int:
    i = 0
    last = -1
    while i < len(lines):
        if not lines[i].startswith('import '):
            break
        j = i
        while j < len(lines):
            t = lines[j].strip()
            if t.startswith('import ') and (' from ' in t or t.startswith("import '") or t.startswith('import "')):
                last = j
                break
            if ' from ' in t and (t.endswith("'") or t.endswith('"') or t.endswith("';") or t.endswith('";')):
                last = j
                break
            j += 1
        i = j + 1
    return last


def inject_styles_var(src: str) -> str:
    pats = [
        re.compile(r"(?:export\s+)?function\s+[A-Za-z0-9_]+\s*\([^)]*\)\s*\{"),
        re.compile(r"(?:export\s+)?const\s+[A-Za-z0-9_]+[^=\n]*=\s*(?:async\s*)?\([^)]*\)\s*=>\s*\{"),
    ]
    ms = sorted([m for p in pats for m in p.finditer(src)], key=lambda x: x.start())

    def close_at(o: int):
        d, j = 0, o
        while j < len(src):
            if src[j] == '{': d += 1
            elif src[j] == '}':
                d -= 1
                if d == 0: return j
            j += 1
        return -1

    ins = []
    for m in ms:
        o = src.find('{', m.start(), m.end())
        c = close_at(o)
        if o == -1 or c == -1: continue
        body = src[o+1:c]
        if 'styles.' not in body: continue
        if re.search(r"\bconst\s+styles\s*=\s*useStyles\(\)", body): continue
        ls = src.rfind('\n', 0, o) + 1
        ind = re.match(r"[ \t]*", src[ls:o]).group(0) + '  '
        ins.append((o+1, '\n' + ind + 'const styles = useStyles();\n'))

    for p, t in sorted(ins, key=lambda x: x[0], reverse=True):
        src = src[:p] + t + src[p:]
    return src


for rel in TARGETS:
    tsx = BASE / f"{rel}.tsx"
    css = BASE / f"{rel}.module.css"
    if not tsx.exists() or not css.exists():
        continue
    src = tsx.read_text(encoding='utf-8').replace('\r\n', '\n')
    css_src = css.read_text(encoding='utf-8')

    src = re.sub(r"^import\s+styles\s+from\s+['\"].*?\.module\.css['\"]\s*$\n?", '', src, flags=re.M)

    lines = src.split('\n')
    end = import_block_end(lines)
    insert_idx = end + 1 if end >= 0 else 0

    if "@fluentui/react-components" not in src:
        lines.insert(insert_idx, "import { makeStyles, tokens } from '@fluentui/react-components'")
        insert_idx += 1

    block_text = make_block(css_src).strip('\n').split('\n')
    lines = lines[:insert_idx+1] + [''] + block_text + [''] + lines[insert_idx+1:]
    src = '\n'.join(lines)

    src = inject_styles_var(src)

    tsx.write_text(src + ('\n' if not src.endswith('\n') else ''), encoding='utf-8')
    css.unlink()

if not list(BASE.rglob('*.module.css')):
    dts = BASE / 'css-modules.d.ts'
    if dts.exists():
        dts.unlink()

print('done')

done


In [5]:
# Correct insertion index bug and rerun migration on a clean restore
restore = []
for rel in TARGETS:
    restore.append(str((BASE / f"{rel}.tsx").relative_to(REPO)).replace('\\\\', '/'))
    restore.append(str((BASE / f"{rel}.module.css").relative_to(REPO)).replace('\\\\', '/'))
restore.append(str((BASE / "css-modules.d.ts").relative_to(REPO)).replace('\\\\', '/'))
subprocess.run(["git", "restore", "--"] + restore, cwd=str(REPO), check=True)

for rel in TARGETS:
    tsx = BASE / f"{rel}.tsx"
    css = BASE / f"{rel}.module.css"
    if not tsx.exists() or not css.exists():
        continue
    src = tsx.read_text(encoding='utf-8').replace('\r\n', '\n')
    css_src = css.read_text(encoding='utf-8')

    src = re.sub(r"^import\s+styles\s+from\s+['\"].*?\.module\.css['\"]\s*$\n?", '', src, flags=re.M)
    lines = src.split('\n')

    insert_idx = import_block_end(lines) + 1
    if "@fluentui/react-components" not in src:
        lines.insert(insert_idx, "import { makeStyles, tokens } from '@fluentui/react-components'")

    block_text = make_block(css_src).strip('\n').split('\n')
    lines = lines[:insert_idx + 1] + [''] + block_text + [''] + lines[insert_idx + 1:]
    src = '\n'.join(lines)
    src = inject_styles_var(src)

    tsx.write_text(src + ('\n' if not src.endswith('\n') else ''), encoding='utf-8')
    css.unlink()

if not list(BASE.rglob('*.module.css')):
    dts = BASE / 'css-modules.d.ts'
    if dts.exists():
        dts.unlink()

print('corrected')

corrected


In [6]:
# Repair pass: ensure `const styles = useStyles();` exists in component body when styles.* is used
fixed = []
for rel in TARGETS:
    tsx = BASE / f"{rel}.tsx"
    if not tsx.exists():
        continue
    src = tsx.read_text(encoding='utf-8').replace('\r\n', '\n')
    if 'styles.' not in src or 'const styles = useStyles();' in src:
        continue

    m = re.search(r"export\s+function\s+[A-Za-z0-9_]+\s*\([^)]*\)\s*\{", src)
    if not m:
        m = re.search(r"export\s+const\s+[A-Za-z0-9_]+[^=\n]*=\s*(?:async\s*)?\([^)]*\)\s*=>\s*\{", src)
    if not m:
        continue

    brace = src.find('{', m.start(), m.end())
    line_start = src.rfind('\n', 0, brace) + 1
    indent = re.match(r"[ \t]*", src[line_start:brace]).group(0) + '  '
    insert = '\n' + indent + 'const styles = useStyles();\n'
    src = src[:brace + 1] + insert + src[brace + 1:]

    tsx.write_text(src + ('\n' if not src.endswith('\n') else ''), encoding='utf-8')
    fixed.append(rel)

print('fixed styles var:', len(fixed))
for rel in fixed:
    print(' -', rel)

fixed styles var: 33
 - pages/ConstraintsPanel
 - pages/KitchenAssistantPanel
 - components/editor/StatusBar
 - components/editor/CompassOverlay
 - components/editor/LockPanel
 - components/editor/ProtectPanel
 - components/editor/GroupsPanel
 - components/editor/MacrosPanel
 - components/editor/LayoutSheetTabs
 - components/editor/LevelsPanel
 - components/editor/DaylightPanel
 - components/editor/AreasPanel
 - components/editor/CameraPresetPanel
 - components/editor/NavigationSettingsPanel
 - components/editor/RenderEnvironmentPanel
 - components/editor/SectionPanel
 - components/editor/StairsPanel
 - components/editor/VisibilityPanel
 - components/editor/WallFeaturesPanel
 - components/editor/RoomFeaturesPanel
 - components/editor/MaterialPanel
 - components/editor/Preview3D
 - components/editor/CanvasArea
 - components/LanguageSwitcher
 - components/OnboardingWizard
 - components/business/BusinessPanel
 - components/catalog/AssetBrowser
 - components/catalog/AssetImportDialog
 - co

In [7]:
# Repair pass 2: remove misplaced styles var lines and insert in actual component body
fixed2 = []
for rel in TARGETS:
    tsx = BASE / f"{rel}.tsx"
    if not tsx.exists():
        continue

    src = tsx.read_text(encoding='utf-8').replace('\r\n', '\n')
    if 'styles.' not in src:
        continue

    # Remove all existing declarations first (misplaced or duplicated)
    src = re.sub(r"^[ \t]*const\s+styles\s*=\s*useStyles\(\);\s*\n?", '', src, flags=re.M)

    m = re.search(r"export\s+function\s+[A-Za-z0-9_]+\s*\([^)]*\)\s*\{", src, flags=re.S)
    if not m:
        m = re.search(r"export\s+const\s+[A-Za-z0-9_]+[^=\n]*=\s*(?:async\s*)?\([^)]*\)\s*=>\s*\{", src, flags=re.S)
    if not m:
        continue

    body_brace = m.end() - 1
    line_start = src.rfind('\n', 0, body_brace) + 1
    base_indent = re.match(r"[ \t]*", src[line_start:body_brace]).group(0)
    inner_indent = base_indent + '  '

    insertion = '\n' + inner_indent + 'const styles = useStyles();\n'
    src = src[:body_brace + 1] + insertion + src[body_brace + 1:]

    tsx.write_text(src + ('\n' if not src.endswith('\n') else ''), encoding='utf-8')
    fixed2.append(rel)

print('repaired body insertion:', len(fixed2))

repaired body insertion: 37


In [8]:
# Consistency check: styles var exists exactly once and is first statement in exported component body
issues = []

for rel in TARGETS:
    tsx = BASE / f"{rel}.tsx"
    if not tsx.exists():
        issues.append((rel, 'missing tsx'))
        continue

    src = tsx.read_text(encoding='utf-8').replace('\r\n', '\n')
    count = len(re.findall(r"\bconst\s+styles\s*=\s*useStyles\(\);", src))
    if count != 1:
        issues.append((rel, f'styles decl count={count}'))
        continue

    m = re.search(r"export\s+function\s+[A-Za-z0-9_]+\s*\([^)]*\)\s*\{", src, flags=re.S)
    if not m:
        m = re.search(r"export\s+const\s+[A-Za-z0-9_]+[^=\n]*=\s*(?:async\s*)?\([^)]*\)\s*=>\s*\{", src, flags=re.S)
    if not m:
        issues.append((rel, 'no exported component signature found'))
        continue

    open_idx = m.end() - 1
    depth = 0
    close_idx = -1
    for i in range(open_idx, len(src)):
        if src[i] == '{':
            depth += 1
        elif src[i] == '}':
            depth -= 1
            if depth == 0:
                close_idx = i
                break
    if close_idx == -1:
        issues.append((rel, 'unbalanced braces'))
        continue

    body = src[open_idx + 1:close_idx]
    body_wo_leading_ws = body.lstrip()
    if not body_wo_leading_ws.startswith('const styles = useStyles();'):
        issues.append((rel, 'styles is not first body statement'))

print('issues:', len(issues))
for rel, msg in issues:
    print(' -', rel, '|', msg)

issues: 0


In [12]:
# Validation via cmd /c to resolve npx on Windows
frontend = BASE.parent

tsc_proc = subprocess.run(
    ['cmd', '/c', 'npx', 'tsc', '--noEmit'],
    cwd=str(frontend),
    capture_output=True,
    text=True,
)

vitest_proc = subprocess.run(
    ['cmd', '/c', 'npx', 'vitest', 'run'],
    cwd=str(frontend),
    capture_output=True,
    text=True,
)

print('tsc exit:', tsc_proc.returncode)
print('vitest exit:', vitest_proc.returncode)

# Print only tail to keep output compact
print('--- tsc tail ---')
print('\n'.join((tsc_proc.stdout + '\n' + tsc_proc.stderr).strip().splitlines()[-40:]))
print('--- vitest tail ---')
print('\n'.join((vitest_proc.stdout + '\n' + vitest_proc.stderr).strip().splitlines()[-60:]))

tsc exit: 0
vitest exit: 0
--- tsc tail ---

--- vitest tail ---
 RUN  v3.2.4 C:/Users/DanielKlas/OneDrive - Tischlermeister Daniel Klas/2_AREAS/AREA_IT_Infrastruktur/Entwicklung/YAKDS/planner-frontend

 âœ“ src/components/editor/autoDollhouse.test.ts (7 tests) 9ms
 âœ“ src/api/runtimeContext.test.ts (9 tests) 11ms
 âœ“ src/editor/actionStateResolver.test.ts (14 tests) 13ms
 âœ“ src/editor/snapUtils.test.ts (15 tests) 13ms
 âœ“ src/api/mediaCapture.test.ts (3 tests) 10ms
 âœ“ src/components/editor/navigationSettings.test.ts (4 tests) 7ms
 âœ“ src/components/editor/renderEnvironmentState.test.ts (7 tests) 11ms
 âœ“ src/components/editor/screenshotCapture.test.ts (6 tests) 11ms
 âœ“ src/pages/plannerViewSettings.test.ts (12 tests) 10ms
 âœ“ src/editor/roomTopology.test.ts (6 tests) 15ms
 âœ“ src/editor/usePolygonEditor.test.js (12 tests) 20ms
 âœ“ src/editor/appShellState.test.ts (5 tests) 10ms
 âœ“ src/components/ribbon/ribbonStateResolver.test.ts (24 tests) 29ms
 âœ“ src/editor/editorP

In [10]:
# Repair pass 3: insert styles hook in all local functions that use styles.*, fix invalid token names, remove unused tokens import
updated = []

for rel in TARGETS:
    tsx = BASE / f"{rel}.tsx"
    if not tsx.exists():
        continue

    src = tsx.read_text(encoding='utf-8').replace('\r\n', '\n')
    original = src

    # Fix known invalid Fluent tokens from migration mapping
    src = src.replace('tokens.colorPaletteBlueForeground1', 'tokens.colorPaletteBlueForeground2')
    src = src.replace('tokens.colorPaletteBlueBackground1', 'tokens.colorPaletteBlueBackground2')
    src = src.replace('tokens.colorPaletteGreenBackground3Hover', 'tokens.colorPaletteGreenBackground3')

    # Insert `const styles = useStyles();` for every function body that uses styles.*
    patterns = [
        re.compile(r"(?:export\s+)?function\s+[A-Za-z0-9_]+\s*\([^)]*\)\s*\{", flags=re.S),
        re.compile(r"(?:export\s+)?(?:const|let|var)\s+[A-Za-z0-9_]+[^=\n]*=\s*(?:async\s*)?\([^)]*\)\s*=>\s*\{", flags=re.S),
    ]

    matches = []
    for p in patterns:
        matches.extend(p.finditer(src))
    matches = sorted(matches, key=lambda m: m.start())

    inserts = []
    for m in matches:
        open_idx = m.end() - 1
        depth = 0
        close_idx = -1
        for i in range(open_idx, len(src)):
            if src[i] == '{':
                depth += 1
            elif src[i] == '}':
                depth -= 1
                if depth == 0:
                    close_idx = i
                    break
        if close_idx == -1:
            continue

        body = src[open_idx + 1:close_idx]
        if 'styles.' not in body:
            continue
        if re.search(r"\bconst\s+styles\s*=\s*useStyles\(\);", body):
            continue

        line_start = src.rfind('\n', 0, open_idx) + 1
        fn_indent = re.match(r"[ \t]*", src[line_start:open_idx]).group(0)
        inner_indent = fn_indent + '  '
        inserts.append((open_idx + 1, '\n' + inner_indent + 'const styles = useStyles();\n'))

    for pos, text in sorted(inserts, key=lambda x: x[0], reverse=True):
        src = src[:pos] + text + src[pos:]

    # Remove unused tokens import in files that don't reference tokens anymore
    if 'tokens.' not in src:
        src = src.replace(
            "import { makeStyles, tokens } from '@fluentui/react-components'",
            "import { makeStyles } from '@fluentui/react-components'",
        )

    if src != original:
        tsx.write_text(src + ('\n' if not src.endswith('\n') else ''), encoding='utf-8')
        updated.append(rel)

print('updated files:', len(updated))
for rel in updated:
    print(' -', rel)

updated files: 6
 - pages/CaptureDialogHarnessPage
 - components/editor/RoomFeaturesPanel
 - components/LanguageSwitcher
 - components/imports/ImportReviewPanel
 - components/validation/ValidationPanel
 - editor/PolygonEditor
